# 📚 Smart Library — Couche Bronze
## Notebook 01 : Ingestion des données brutes vers Apache Iceberg

---

### Objectif
Ce notebook ingère **toutes les sources de données brutes** vers la couche Bronze du Data Lakehouse.
Les données sont stockées dans **MinIO** (Data Lake) sous format **Apache Iceberg**, catalogué via
**Lakekeeper** (catalogue REST, métadonnées persistées dans **PostgreSQL**).

### Sources traitées
| Source | Format | Table Bronze | Mode Iceberg |
|--------|--------|--------------|----------|
| Utilisateurs | CSV | `bronze.utilisateurs` | copy-on-write |
| Livres | JSON | `bronze.livres` | copy-on-write |
| Catalogue | XML | `bronze.catalogue` | copy-on-write |
| Commandes | CSV | `bronze.commandes` | copy-on-write |
| Stock (COW) | CSV | `bronze.stock_cow` | copy-on-write |
| Stock (MOR) | CSV | `bronze.stock_mor` | merge-on-read |
| Factures | TXT | `bronze.factures` | copy-on-write |

> 💡 **COW vs MOR sous Iceberg** : contrairement à Hudi où COW/MOR est un *type de table* déclaré
> à l'écriture, en Iceberg c'est une **propriété de table** (`write.update.mode`, `write.delete.mode`,
> `write.merge.mode`) fixée à la création. Le stock est ingéré dans les deux modes pour comparer
> leur comportement.

---
## Étape 0 — Initialisation de la SparkSession (Iceberg + Lakekeeper)

In [1]:
import os, sys, time, re
from datetime import datetime
from pathlib import Path

# ── Tuer toute session existante ─────────────────────────────
try:
    from pyspark.sql import SparkSession
    SparkSession.getActiveSession().stop()
except:
    pass

# ── Java ──────────────────────────────────────────────────────
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"]      = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

# ── Jars Iceberg + MinIO ──────────────────────────────────────
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.5.2,"
    "org.apache.iceberg:iceberg-aws-bundle:1.5.2,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Connexion MinIO + Lakekeeper ──────────────────────────────
MINIO_ENDPOINT       = "http://minio:9000"
MINIO_ACCESS         = "minioadmin"
MINIO_SECRET         = "minioadmin123"
CATALOG_NAME         = "lakehouse"
LAKEKEEPER_URI       = "http://lakekeeper:8181/catalog"
LAKEKEEPER_WAREHOUSE = "lakehouse"

NS_BRONZE = f"{CATALOG_NAME}.bronze"
NS_SILVER = f"{CATALOG_NAME}.silver"
NS_GOLD   = f"{CATALOG_NAME}.gold"

# ── Fonctions utilitaires Iceberg (équivalent hudi_options) ───
#
#  Avec Hudi : le type de table (COW/MOR) et la clé d'upsert se
#  déclarent au niveau de l'ÉCRITURE (options du DataFrameWriter).
#  Avec Iceberg : ce sont des PROPRIÉTÉS DE TABLE, et l'UPSERT
#  s'exprime explicitement en SQL via MERGE INTO.
#
def iceberg_table_properties(type_table="copy-on-write"):
    """
    type_table :
      - "copy-on-write" → équivalent Hudi COW
      - "merge-on-read"  → équivalent Hudi MOR
    """
    return {
        "format-version"    : "2",
        "write.delete.mode" : type_table,
        "write.update.mode" : type_table,
        "write.merge.mode"  : type_table,
    }

def create_or_replace_table(df, table_name, partition_cols=None, type_table="copy-on-write"):
    """Équivalent de : df.write.format('hudi')....mode('overwrite').save(chemin)"""
    # Iceberg (ClusteredWriter) exige que les lignes soient triées/regroupées
    # par colonne de partition AVANT l'écriture — contrairement à Hudi qui
    # gère le clustering en interne. Sans ce tri : "Incoming records violate
    # the writer assumption that records are clustered by spec and partition".
    if partition_cols:
        df = df.sort(*partition_cols)
    writer = df.writeTo(table_name)
    if partition_cols:
        writer = writer.partitionedBy(*[F.col(c) for c in partition_cols])
    for k, v in iceberg_table_properties(type_table).items():
        writer = writer.tableProperty(k, v)
    writer.createOrReplace()

def append_table(df, table_name):
    """Ajout de lignes (insert simple, pas d'upsert)."""
    df.writeTo(table_name).append()

def upsert_table(spark, df, table_name, key_cols):
    """UPSERT via MERGE INTO — équivalent fonctionnel de l'UPSERT automatique Hudi."""
    vue = f"_staging_{table_name.split('.')[-1]}"
    df.createOrReplaceTempView(vue)
    condition = " AND ".join([f"t.{c} = s.{c}" for c in key_cols])
    spark.sql(f"""
        MERGE INTO {table_name} t
        USING {vue} s
        ON {condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

def snapshots(spark, table_name):
    """Retourne l'historique des snapshots Iceberg (équivalent de la timeline Hudi)."""
    return spark.sql(f"SELECT * FROM {table_name}.snapshots ORDER BY committed_at")

# ── SparkSession ──────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("SmartLibrary - Ingestion Bronze")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", LAKEKEEPER_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", LAKEKEEPER_WAREHOUSE)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint",           MINIO_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.access-key-id",     MINIO_ACCESS)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.secret-access-key", MINIO_SECRET)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .config("spark.serializer",
            "org.apache.spark.serializer.KryoSerializer")
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ── Création des namespaces Medallion (idempotent) ────────────
for ns in (NS_BRONZE, NS_SILVER, NS_GOLD):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {ns}")

print(f"✅ SparkSession démarrée — version {spark.version}")
print(f"   Catalogue Iceberg : {CATALOG_NAME} (REST → Lakekeeper → Postgres)")
print(f"   Stockage          : {MINIO_ENDPOINT} (bucket lakehouse)")

✅ SparkSession démarrée — version 3.4.0
   Catalogue Iceberg : lakehouse (REST → Lakekeeper → Postgres)
   Stockage          : http://minio:9000 (bucket lakehouse)


---
## Étape 1 — Ingestion des Utilisateurs (CSV → Iceberg, copy-on-write)

In [2]:
df_utilisateurs = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "UTF-8")
    .csv("/home/jovyan/data/batch/utilisateurs.csv")
)

# Renommer user_id → utilisateur_id si nécessaire
if "user_id" in df_utilisateurs.columns:
    df_utilisateurs = df_utilisateurs.withColumnRenamed("user_id", "utilisateur_id")

print(f"📋 Utilisateurs chargés : {df_utilisateurs.count()} lignes")
print(f"   Colonnes : {df_utilisateurs.columns}")

📋 Utilisateurs chargés : 1000 lignes
   Colonnes : ['utilisateur_id', 'nom', 'prenom', 'email', 'telephone', 'ville', 'pays', 'date_inscription', 'genre_prefere', 'nb_commandes_total', 'statut', 'updated_at']


In [3]:
df_utilisateurs_bronze = (
    df_utilisateurs
    .withColumn("updated_at",      F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("_source",         F.lit("batch/utilisateurs.csv"))
    .withColumn("_ingestion_date", F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("_couche",         F.lit("bronze"))
    .withColumn("partition_pays",  F.col("pays"))
)

df_utilisateurs_bronze.select(
    "utilisateur_id", "nom", "email", "pays", "partition_pays", "_couche"
).show(5, truncate=False)

+--------------+-------+----------------------------+-------+--------------+-------+
|utilisateur_id|nom    |email                       |pays   |partition_pays|_couche|
+--------------+-------+----------------------------+-------+--------------+-------+
|USR00001      |Peltier|denisguerin@example.com     |Maroc  |Maroc         |bronze |
|USR00002      |Mallet |margotbouvier@example.org   |Tunisie|Tunisie       |bronze |
|USR00003      |Gimenez|duhamelmichele@example.com  |Maroc  |Maroc         |bronze |
|USR00004      |Bertin |martin66@example.com        |Tunisie|Tunisie       |bronze |
|USR00005      |Lacroix|isabellepasquier@example.org|Maroc  |Maroc         |bronze |
+--------------+-------+----------------------------+-------+--------------+-------+
only showing top 5 rows



In [4]:
TABLE_UTILISATEURS = f"{NS_BRONZE}.utilisateurs"

create_or_replace_table(
    df_utilisateurs_bronze,
    TABLE_UTILISATEURS,
    partition_cols=["partition_pays"],
    type_table="copy-on-write"
)

n = spark.table(TABLE_UTILISATEURS).count()
print(f"✅ {TABLE_UTILISATEURS} — copy-on-write — {n} lignes")

✅ lakehouse.bronze.utilisateurs — copy-on-write — 1000 lignes


---
## Étape 2 — Ingestion des Livres (JSON → Iceberg, copy-on-write)

In [5]:
df_livres = (
    spark.read
    .option("multiline", "true")
    .option("encoding", "UTF-8")
    .json("/home/jovyan/data/batch/livres.json")
)

if "book_id" in df_livres.columns:
    df_livres = df_livres.withColumnRenamed("book_id", "livre_id")

print(f"📚 Livres chargés : {df_livres.count()} livres")

📚 Livres chargés : 500 livres


In [6]:
df_livres_bronze = (
    df_livres
    .withColumn("tags_str", F.when(
        F.col("tags").isNotNull(),
        F.array_join(F.col("tags"), ",")
    ).otherwise(F.lit("")))
    .drop("tags")
    .withColumn("updated_at",      F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("_source",         F.lit("batch/livres.json"))
    .withColumn("_ingestion_date", F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("_couche",         F.lit("bronze"))
    .withColumn("partition_genre", F.col("genre"))
)

df_livres_bronze.select(
    "livre_id", "titre", "auteur", "genre", "prix_vente"
).show(5, truncate=True)

+--------+--------------------+--------------------+----------+----------+
|livre_id|               titre|              auteur|     genre|prix_vente|
+--------+--------------------+--------------------+----------+----------+
|  BK0001|Étudier Parfaitement|       Michelle Ruiz|Biographie|     48.48|
|  BK0002|Mensonge Explique...|         Alex Nguyen|  Policier|     14.14|
|  BK0003|Colon Tomber Cher...|      Margaud Pichon|    Poésie|     14.51|
|  BK0004|Approcher Premier...|    Timothée Lefèvre|  Jeunesse|     30.75|
|  BK0005|Accepter Demander...|Denis Arnaud du A...|     Roman|     35.73|
+--------+--------------------+--------------------+----------+----------+
only showing top 5 rows



In [7]:
TABLE_LIVRES = f"{NS_BRONZE}.livres"

create_or_replace_table(
    df_livres_bronze,
    TABLE_LIVRES,
    partition_cols=["partition_genre"],
    type_table="copy-on-write"
)

n = spark.table(TABLE_LIVRES).count()
print(f"✅ {TABLE_LIVRES} — copy-on-write — {n} lignes")

✅ lakehouse.bronze.livres — copy-on-write — 500 lignes


---
## Étape 3 — Ingestion du Catalogue (XML → Iceberg, copy-on-write)

In [8]:
import xml.etree.ElementTree as ET

tree   = ET.parse("/home/jovyan/data/batch/catalogue.xml")
racine = tree.getroot()
lignes_catalogue = []

for categorie in racine.findall("categorie"):
    nom_cat = categorie.get("nom", "")
    id_cat  = categorie.get("id", "")
    desc    = categorie.findtext("description", "")
    for sous_cat in categorie.findall("sous_categorie"):
        nom_scat = sous_cat.get("nom", "")
        for livre in sous_cat.findall("livre"):
            livre_id = livre.get("id", "")
            lignes_catalogue.append({
                "catalogue_id"       : f"{id_cat}_{nom_scat[:6]}_{livre_id}",
                "livre_id"           : livre_id,
                "categorie_id"       : id_cat,
                "nom_categorie"      : nom_cat,
                "nom_sous_categorie" : nom_scat,
                "description_cat"    : desc,
                "ordre_affichage"    : int(livre.findtext("ordre_affichage", "0")),
                "en_vedette"         : livre.findtext("en_vedette", "false") == "true",
                "updated_at"         : datetime.now().isoformat(),
                "_source"            : "batch/catalogue.xml",
                "_ingestion_date"    : datetime.now().isoformat(),
                "_couche"            : "bronze",
                "partition_cat"      : nom_cat,
            })

df_catalogue_bronze = (
    spark.createDataFrame(lignes_catalogue)
    .withColumn("updated_at",      F.col("updated_at").cast(StringType()))
    .withColumn("_ingestion_date", F.col("_ingestion_date").cast(StringType()))
)

print(f"📂 Catalogue parsé : {df_catalogue_bronze.count()} associations")
df_catalogue_bronze.select(
    "catalogue_id", "livre_id", "nom_categorie", "nom_sous_categorie"
).show(5, truncate=False)

📂 Catalogue parsé : 304 associations
+-------------------+--------+-------------+------------------+
|catalogue_id       |livre_id|nom_categorie|nom_sous_categorie|
+-------------------+--------+-------------+------------------+
|roman_Roman _BK0036|BK0036  |Roman        |Roman classique   |
|roman_Roman _BK0086|BK0086  |Roman        |Roman classique   |
|roman_Roman _BK0096|BK0096  |Roman        |Roman classique   |
|roman_Roman _BK0171|BK0171  |Roman        |Roman classique   |
|roman_Roman _BK0461|BK0461  |Roman        |Roman classique   |
+-------------------+--------+-------------+------------------+
only showing top 5 rows



In [9]:
TABLE_CATALOGUE = f"{NS_BRONZE}.catalogue"

create_or_replace_table(
    df_catalogue_bronze,
    TABLE_CATALOGUE,
    partition_cols=["partition_cat"],
    type_table="copy-on-write"
)

n = spark.table(TABLE_CATALOGUE).count()
print(f"✅ {TABLE_CATALOGUE} — copy-on-write — {n} lignes")

✅ lakehouse.bronze.catalogue — copy-on-write — 304 lignes


---
## Étape 4 — Ingestion des Commandes (CSV → Iceberg, copy-on-write)

In [10]:
df_commandes = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "UTF-8")
    .csv("/home/jovyan/data/batch/commandes.csv")
)

# Renommer colonnes anglaises si nécessaire
for ancien, nouveau in {"order_id": "commande_id", "user_id": "utilisateur_id", "book_id": "livre_id"}.items():
    if ancien in df_commandes.columns:
        df_commandes = df_commandes.withColumnRenamed(ancien, nouveau)

print(f"🛒 Commandes chargées : {df_commandes.count()} lignes")
df_commandes.groupBy("statut").count().orderBy("count", ascending=False).show()

🛒 Commandes chargées : 5000 lignes
+---------+-----+
|   statut|count|
+---------+-----+
|    livré| 3268|
| en_cours|  997|
|   annulé|  484|
|remboursé|  251|
+---------+-----+



In [11]:
df_commandes_bronze = (
    df_commandes
    .withColumn("date_commande_ts",  F.to_timestamp("date_commande"))
    .withColumn("annee",             F.year("date_commande_ts").cast(StringType()))
    .withColumn("mois",              F.lpad(F.month("date_commande_ts").cast(StringType()), 2, "0"))
    .withColumn("partition_periode", F.concat_ws("-", F.col("annee"), F.col("mois")))
    .withColumn("updated_at",        F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("_source",           F.lit("batch/commandes.csv"))
    .withColumn("_ingestion_date",   F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("_couche",           F.lit("bronze"))
)

df_commandes_bronze.select(
    "commande_id", "utilisateur_id", "livre_id",
    "montant_total", "statut", "partition_periode"
).show(5, truncate=False)

+-----------+--------------+--------+-------------+--------+-----------------+
|commande_id|utilisateur_id|livre_id|montant_total|statut  |partition_periode|
+-----------+--------------+--------+-------------+--------+-----------------+
|ORD0000001 |USR00294      |BK0175  |102.0        |en_cours|2025-03          |
|ORD0000002 |USR00152      |BK0295  |107.32       |livré   |2024-09          |
|ORD0000003 |USR00898      |BK0233  |91.96        |annulé  |2024-12          |
|ORD0000004 |USR00568      |BK0365  |82.12        |livré   |2026-03          |
|ORD0000005 |USR00025      |BK0354  |43.34        |livré   |2026-02          |
+-----------+--------------+--------+-------------+--------+-----------------+
only showing top 5 rows



In [12]:
TABLE_COMMANDES = f"{NS_BRONZE}.commandes"

create_or_replace_table(
    df_commandes_bronze,
    TABLE_COMMANDES,
    partition_cols=["partition_periode"],
    type_table="copy-on-write"
)

n = spark.table(TABLE_COMMANDES).count()
print(f"✅ {TABLE_COMMANDES} — copy-on-write — {n} lignes")
print(f"   Partitions : {df_commandes_bronze.select('partition_periode').distinct().count()} périodes")

✅ lakehouse.bronze.commandes — copy-on-write — 5000 lignes
   Partitions : 25 périodes


---
## Étape 5 — Ingestion du Stock : copy-on-write et merge-on-read (comparaison)

> 💡 **Démonstration clé** :
> - `bronze.stock_cow` → **copy-on-write** : chaque UPDATE réécrit intégralement les fichiers
>   de données Parquet concernés (nouveau snapshot avec de nouveaux data files).
> - `bronze.stock_mor` → **merge-on-read** : les UPDATE/DELETE sont écrits sous forme de
>   **delete files** (positional/equality deletes) légers, fusionnés à la lecture avec les
>   data files existants — l'équivalent Iceberg des fichiers `.log` Avro de Hudi MOR.

In [13]:
df_stock = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "UTF-8")
    .csv("/home/jovyan/data/batch/inventaire.csv")
)

if "book_id" in df_stock.columns:
    df_stock = df_stock.withColumnRenamed("book_id", "livre_id")

df_stock_bronze = (
    df_stock
    .withColumn("stock_total",
                F.col("quantite_disponible") + F.col("quantite_reservee"))
    .withColumn("alerte_stock",
                (F.col("quantite_disponible") <= F.col("seuil_alerte")).cast("boolean"))
    .withColumn("updated_at",         F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("_source",            F.lit("batch/inventaire.csv"))
    .withColumn("_ingestion_date",    F.lit(datetime.now().isoformat()).cast(StringType()))
    .withColumn("_couche",            F.lit("bronze"))
    .withColumn("partition_entrepot", F.col("entrepot"))
)

print(f"📦 Stock chargé : {df_stock_bronze.count()} lignes")
print(f"   Livres en alerte : {df_stock_bronze.filter(F.col('alerte_stock') == True).count()}")
df_stock_bronze.select(
    "stock_id", "livre_id", "quantite_disponible", "stock_total", "alerte_stock"
).show(5, truncate=False)

📦 Stock chargé : 500 lignes
   Livres en alerte : 24
+----------+--------+-------------------+-----------+------------+
|stock_id  |livre_id|quantite_disponible|stock_total|alerte_stock|
+----------+--------+-------------------+-----------+------------+
|STK-BK0001|BK0001  |44                 |59         |false       |
|STK-BK0002|BK0002  |65                 |68         |false       |
|STK-BK0003|BK0003  |86                 |106        |false       |
|STK-BK0004|BK0004  |41                 |45         |false       |
|STK-BK0005|BK0005  |116                |131        |false       |
+----------+--------+-------------------+-----------+------------+
only showing top 5 rows



In [14]:
# ── Écriture copy-on-write ────────────────────────────────────
TABLE_STOCK_COW = f"{NS_BRONZE}.stock_cow"

debut = time.time()
create_or_replace_table(
    df_stock_bronze, TABLE_STOCK_COW,
    partition_cols=["partition_entrepot"], type_table="copy-on-write"
)
temps_ecriture_cow = round(time.time() - debut, 2)
print(f"✅ {TABLE_STOCK_COW} — copy-on-write — créée en {temps_ecriture_cow}s")

# ── Écriture merge-on-read ────────────────────────────────────
TABLE_STOCK_MOR = f"{NS_BRONZE}.stock_mor"

debut = time.time()
create_or_replace_table(
    df_stock_bronze, TABLE_STOCK_MOR,
    partition_cols=["partition_entrepot"], type_table="merge-on-read"
)
temps_ecriture_mor = round(time.time() - debut, 2)
print(f"✅ {TABLE_STOCK_MOR} — merge-on-read — créée en {temps_ecriture_mor}s")

✅ lakehouse.bronze.stock_cow — copy-on-write — créée en 0.38s
✅ lakehouse.bronze.stock_mor — merge-on-read — créée en 0.38s


In [15]:
# ── Simulation : 50 mises à jour → comparaison COW vs MOR ────
# Avec Iceberg, la mise à jour se fait via MERGE INTO (pas d'upsert
# implicite comme avec Hudi) : c'est le mode de table (copy-on-write /
# merge-on-read) qui détermine COMMENT le moteur applique ce MERGE.

df_maj_cow = (
    spark.table(TABLE_STOCK_COW).limit(50)
    .withColumn("quantite_disponible", F.col("quantite_disponible") + F.lit(10))
    .withColumn("updated_at", F.lit(datetime.now().isoformat()).cast(StringType()))
)

df_maj_mor = (
    spark.table(TABLE_STOCK_MOR).limit(50)
    .withColumn("quantite_disponible", F.col("quantite_disponible") + F.lit(10))
    .withColumn("updated_at", F.lit(datetime.now().isoformat()).cast(StringType()))
)

debut = time.time()
upsert_table(spark, df_maj_cow, TABLE_STOCK_COW, key_cols=["stock_id"])
temps_maj_cow = round(time.time() - debut, 2)

debut = time.time()
upsert_table(spark, df_maj_mor, TABLE_STOCK_MOR, key_cols=["stock_id"])
temps_maj_mor = round(time.time() - debut, 2)

print("=" * 55)
print("  RÉSULTATS — 50 mises à jour de stock (MERGE INTO)")
print("=" * 55)
print(f"  Écriture initiale  COW : {temps_ecriture_cow}s")
print(f"  Écriture initiale  MOR : {temps_ecriture_mor}s")
print(f"  Mise à jour (50)   COW : {temps_maj_cow}s  ← réécrit les fichiers Parquet concernés")
print(f"  Mise à jour (50)   MOR : {temps_maj_mor}s  ← écrit des delete files + nouveaux data files")
print()
print("  💡 MOR est généralement plus rapide en écriture car il n'a pas")
print("     besoin de réécrire intégralement les fichiers Parquet existants")

  RÉSULTATS — 50 mises à jour de stock (MERGE INTO)
  Écriture initiale  COW : 0.38s
  Écriture initiale  MOR : 0.38s
  Mise à jour (50)   COW : 1.59s  ← réécrit les fichiers Parquet concernés
  Mise à jour (50)   MOR : 0.87s  ← écrit des delete files + nouveaux data files

  💡 MOR est généralement plus rapide en écriture car il n'a pas
     besoin de réécrire intégralement les fichiers Parquet existants


In [16]:
# ── Comparaison lecture ───────────────────────────────────────
debut = time.time()
n_cow = spark.table(TABLE_STOCK_COW).count()
t_cow = round(time.time() - debut, 2)

# Iceberg fusionne automatiquement delete files + data files à la lecture
# (pas d'option "query type" séparée comme Hudi snapshot/read_optimized ;
#  c'est le comportement par défaut du scan Iceberg)
debut = time.time()
n_mor = spark.table(TABLE_STOCK_MOR).count()
t_mor = round(time.time() - debut, 2)

# Lecture "read-optimized" équivalente : ignorer les delete files
# (via l'option split-open-file-cost n'existe pas nativement ; on peut
#  approcher via un scan sur le snapshot juste avant les MERGE)
snaps_mor = spark.sql(f"SELECT snapshot_id, committed_at, operation FROM {TABLE_STOCK_MOR}.snapshots ORDER BY committed_at")
premier_snapshot_mor = snaps_mor.first()["snapshot_id"]
debut = time.time()
n_mor_initial = spark.read.format("iceberg").option("snapshot-id", premier_snapshot_mor).load(TABLE_STOCK_MOR).count()
t_mor_initial = round(time.time() - debut, 2)

print("=" * 58)
print("  COMPARAISON LECTURE")
print("=" * 58)
print(f"  COW  (data files uniquement)      : {t_cow}s  → {n_cow} lignes")
print(f"  MOR  (data + delete files fusionnés): {t_mor}s  → {n_mor} lignes (données fraîches)")
print(f"  MOR  snapshot initial (VERSION AS OF): {t_mor_initial}s  → {n_mor_initial} lignes")
print()
print("  💡 Vérifie dans MinIO Console → http://localhost:9001")
print("     bronze/stock_cow/data/ → fichiers .parquet uniquement")
print("     bronze/stock_mor/data/ → fichiers .parquet (data) + delete files")

  COMPARAISON LECTURE
  COW  (data files uniquement)      : 0.1s  → 500 lignes
  MOR  (data + delete files fusionnés): 0.28s  → 500 lignes (données fraîches)
  MOR  snapshot initial (VERSION AS OF): 0.09s  → 500 lignes

  💡 Vérifie dans MinIO Console → http://localhost:9001
     bronze/stock_cow/data/ → fichiers .parquet uniquement
     bronze/stock_mor/data/ → fichiers .parquet (data) + delete files


---
## Étape 6 — Ingestion des Factures (TXT simulé PDF → Iceberg, copy-on-write)

In [17]:
dossier_factures = Path("/home/jovyan/data/batch/factures")
lignes_factures  = []

def extraire(pattern, texte, defaut=""):
    match = re.search(pattern, texte)
    return match.group(1).strip() if match else defaut

for fichier in sorted(dossier_factures.glob("*.txt")):
    contenu    = fichier.read_text(encoding="utf-8")
    total_ttc  = extraire(r"TOTAL TTC\s+:\s+([\d.]+)", contenu)
    lignes_factures.append({
        "facture_id"       : extraire(r"FACTURE\s+(\S+)", contenu),
        "client_id"        : extraire(r"Client\s+:\s+(\S+)", contenu),
        "date_facture"     : extraire(r"Date\s+:\s+(\S+)", contenu),
        "montant_ttc"      : float(total_ttc) if total_ttc else 0.0,
        "methode_paiement" : extraire(r"Méthode de paiement\s+:\s+(\S+)", contenu),
        "contenu_brut"     : contenu[:300],
        "nom_fichier"      : fichier.name,
        "updated_at"       : datetime.now().isoformat(),
        "_source"          : "batch/factures/*.txt",
        "_ingestion_date"  : datetime.now().isoformat(),
        "_couche"          : "bronze",
        "partition_statut" : "payé",
    })

df_factures_bronze = (
    spark.createDataFrame(lignes_factures)
    .withColumn("updated_at",      F.col("updated_at").cast(StringType()))
    .withColumn("_ingestion_date", F.col("_ingestion_date").cast(StringType()))
)

print(f"🧾 Factures extraites : {df_factures_bronze.count()}")
df_factures_bronze.select(
    "facture_id", "client_id", "date_facture", "montant_ttc"
).show(5, truncate=False)

🧾 Factures extraites : 50
+----------+---------+------------+-----------+
|facture_id|client_id|date_facture|montant_ttc|
+----------+---------+------------+-----------+
|FAC0001   |USR00758 |2026-06-17  |300.61     |
|FAC0002   |USR00154 |2026-01-25  |101.39     |
|FAC0003   |USR00900 |2026-05-25  |160.61     |
|FAC0004   |USR00966 |2025-08-28  |134.57     |
|FAC0005   |USR00194 |2026-04-30  |110.35     |
+----------+---------+------------+-----------+
only showing top 5 rows



In [18]:
TABLE_FACTURES = f"{NS_BRONZE}.factures"

create_or_replace_table(
    df_factures_bronze,
    TABLE_FACTURES,
    partition_cols=["partition_statut"],
    type_table="copy-on-write"
)

n = spark.table(TABLE_FACTURES).count()
print(f"✅ {TABLE_FACTURES} — copy-on-write — {n} lignes")

✅ lakehouse.bronze.factures — copy-on-write — 50 lignes


---
## Étape 7 — Bilan de la couche Bronze

In [19]:
tables = [
    ("bronze.utilisateurs", TABLE_UTILISATEURS, "COW"),
    ("bronze.livres",       TABLE_LIVRES,        "COW"),
    ("bronze.catalogue",    TABLE_CATALOGUE,     "COW"),
    ("bronze.commandes",    TABLE_COMMANDES,     "COW"),
    ("bronze.stock_cow",    TABLE_STOCK_COW,     "COW"),
    ("bronze.stock_mor",    TABLE_STOCK_MOR,     "MOR"),
    ("bronze.factures",     TABLE_FACTURES,      "COW"),
]

print("=" * 60)
print(f"  {'TABLE':<28} {'MODE':<5} {'LIGNES':>8}")
print("=" * 60)
total = 0
for nom, table, type_t in tables:
    try:
        n = spark.table(table).count()
        total += n
        print(f"  {nom:<28} {type_t:<5} {n:>8,}  ✅")
    except Exception as e:
        print(f"  {nom:<28} {type_t:<5} {'ERREUR':>8}  ❌")
print("=" * 60)
print(f"  {'TOTAL':<28}       {total:>8,}")
print("=" * 60)
print()
print("  MinIO Console  → http://localhost:9001")
print("  Bucket         → lakehouse/bronze/")
print("  Catalogue REST → http://localhost:8181 (Lakekeeper)")
print()
print("  ➡  Prochain : 02_traitement_silver.ipynb")

  TABLE                        MODE    LIGNES
  bronze.utilisateurs          COW      1,000  ✅
  bronze.livres                COW        500  ✅
  bronze.catalogue             COW        304  ✅
  bronze.commandes             COW      5,000  ✅
  bronze.stock_cow             COW        500  ✅
  bronze.stock_mor             MOR        500  ✅
  bronze.factures              COW         50  ✅
  TOTAL                                 7,854

  MinIO Console  → http://localhost:9001
  Bucket         → lakehouse/bronze/
  Catalogue REST → http://localhost:8181 (Lakekeeper)

  ➡  Prochain : 02_traitement_silver.ipynb


In [20]:
# ── Métadonnées Iceberg (équivalent de la timeline Hudi) ─────
print("=== Snapshots Iceberg sur bronze.commandes ===")
snapshots(spark, TABLE_COMMANDES).select(
    "snapshot_id", "committed_at", "operation"
).show(10, truncate=False)

print("=== Aperçu des données (colonnes métier) ===")
spark.table(TABLE_COMMANDES).select(
    "commande_id", "statut", "montant_total"
).show(5, truncate=False)

print("💡 Les snapshot_id seront utilisés pour le Time Travel dans le notebook 05")
print("   (VERSION AS OF <snapshot_id> / TIMESTAMP AS OF <timestamp>)")

=== Snapshots Iceberg sur bronze.commandes ===
+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|2543600660653937882|2026-07-13 23:12:32.238|append   |
+-------------------+-----------------------+---------+

=== Aperçu des données (colonnes métier) ===
+-----------+--------+-------------+
|commande_id|statut  |montant_total|
+-----------+--------+-------------+
|ORD0000019 |en_cours|27.66        |
|ORD0000222 |livré   |39.5         |
|ORD0000229 |annulé  |47.28        |
|ORD0000243 |livré   |32.82        |
|ORD0000288 |annulé  |31.57        |
+-----------+--------+-------------+
only showing top 5 rows

💡 Les snapshot_id seront utilisés pour le Time Travel dans le notebook 05
   (VERSION AS OF <snapshot_id> / TIMESTAMP AS OF <timestamp>)


In [21]:
spark.stop()
print("✅ SparkSession fermée — Couche Bronze complète !")

✅ SparkSession fermée — Couche Bronze complète !
